In [121]:
!pip install tabulate


In [122]:
import os
import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
sns.set_context("talk")


In [123]:
OUTDIR = Path("output")
OUTDIR.mkdir(parents=True, exist_ok=True)

HIGH_MISSING_THRESHOLD = 0.40   # 40% missing => flag as high-missing
TOP_K_CATS = 15                 # frequency table top categories
MAX_CATEGORICAL_UNIQUES = 25    # choose cat col with <= 25 uniques when possible
MAX_TEXT_UNIQUES_FRAC = 0.50    # if nunique > 50% rows => treat as text-like
DATETIME_PARSE_THRESHOLD = 0.80 # if >=80% parseable => datetime

# Any number of datasets:
DATASETS = [
    {"name": "data2", "csv": "data2.csv", "schema": None},
    {"name": "chocolatesales", "csv": "chocolatesales.csv", "schema": None},
    # {"name": "unseen_1", "csv": "data/unseen.csv", "schema": "data/schema_unseen.csv"},
    # {"name": "unseen_2", "csv": "data/another.csv", "schema": None},
]

for d in DATASETS:
    print(d["name"], "CSV exists?", Path(d["csv"]).exists(), "| schema:", d.get("schema"))


data2 CSV exists? True | schema: None
chocolatesales CSV exists? True | schema: None


In [124]:
def load_schema(schema_path: str | None):
    """
    Optional schema/data dictionary.
    Accepts CSV or JSON.
    CSV recommended columns: column, description, unit, allowed_values, missing_codes
    JSON recommended structure: { "columns": { "colname": {...} } }
    """
    if not schema_path:
        return None

    p = Path(schema_path)
    if not p.exists():
        return None

    if p.suffix.lower() == ".csv":
        s = pd.read_csv(p)
        s.columns = [c.strip().lower() for c in s.columns]
        if "column" not in s.columns and "name" in s.columns:
            s = s.rename(columns={"name": "column"})
        if "column" not in s.columns:
            return {"type": "csv", "raw_rows": s.to_dict(orient="records")}

        s["column"] = s["column"].astype(str)
        return {"type": "csv", "rows": s.to_dict(orient="records")}

    if p.suffix.lower() == ".json":
        return {"type": "json", **json.loads(p.read_text(encoding="utf-8"))}

    return None


def schema_missing_codes(schema_obj):
    """Extract missing codes from schema if present."""
    if not schema_obj:
        return []
    codes = []
    if schema_obj.get("type") == "csv" and "rows" in schema_obj:
        for r in schema_obj["rows"]:
            # common column names
            for key in ["missing_codes", "missing", "missing_values", "na_values"]:
                if key in r and pd.notna(r[key]):
                    # split by comma/semicolon
                    parts = re.split(r"[;,]\s*", str(r[key]).strip())
                    codes.extend([p for p in parts if p != ""])
    if schema_obj.get("type") == "json":
        cols = schema_obj.get("columns", {})
        for col, meta in cols.items():
            for key in ["missing_codes", "missing", "na_values"]:
                val = meta.get(key)
                if val:
                    if isinstance(val, list):
                        codes.extend(val)
                    else:
                        parts = re.split(r"[;,]\s*", str(val).strip())
                        codes.extend([p for p in parts if p != ""])
    return codes


In [125]:
DEFAULT_MISSING_CODES = {
    "", "na", "n/a", "null", "none", "nan", "unknown", "unk", "?", "-", "--", " ", "missing",
    "-999", "-9999", "999", "9999"
}

def read_csv_robust(csv_path: str):
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    return df

def apply_missing_codes(df: pd.DataFrame, extra_missing_codes=None):
    missing_codes = set(DEFAULT_MISSING_CODES)
    if extra_missing_codes:
        missing_codes |= set(map(str, extra_missing_codes))

    # normalize object columns (strip + casefold)
    for c in df.columns:
        if df[c].dtype == "object":
            s = df[c].astype(str).str.strip()
            # replace exact codes (case-insensitive)
            lower_map = {str(v).strip().lower(): np.nan for v in missing_codes}
            df[c] = s.apply(lambda x: np.nan if str(x).strip().lower() in lower_map else x)

    # If an object column is almost entirely numeric strings, convert it
    for c in df.columns:
        if df[c].dtype == "object":
            maybe_num = pd.to_numeric(df[c], errors="coerce")
            if maybe_num.notna().mean() > 0.95:
                df[c] = maybe_num

    return df

def infer_column_roles(df: pd.DataFrame):
    roles = {}
    n = len(df)

    for c in df.columns:
        s = df[c]

        # numeric
        if pd.api.types.is_numeric_dtype(s):
            roles[c] = "numeric"
            continue

        # datetime attempt for object
        if s.dtype == "object":
            parsed = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() >= DATETIME_PARSE_THRESHOLD:
                roles[c] = "datetime"
                df[c] = parsed
                continue

            nunique = s.nunique(dropna=True)
            # text-like (very high cardinality)
            if (n > 0) and (nunique > max(50, MAX_TEXT_UNIQUES_FRAC * n)):
                roles[c] = "text"
            else:
                roles[c] = "categorical"
        else:
            roles[c] = "categorical"

    return roles, df


In [126]:
def detect_identifier_like_columns(df: pd.DataFrame):
    id_like = []
    n = len(df)
    if n == 0:
        return id_like

    for c in df.columns:
        s = df[c]
        nunique = s.nunique(dropna=True)
        name_hint = any(k in c.lower() for k in ["id", "ssn", "email", "phone", "uuid", "user", "account"])

        # high uniqueness is the main signal; name hints make it stronger
        if nunique >= 0.95 * n and (name_hint or s.dtype == "object"):
            id_like.append(c)

    return id_like

def dataset_quality_checks(df: pd.DataFrame, roles: dict, high_missing_thr=0.40):
    n_rows, n_cols = df.shape

    missing_pct = df.isna().mean().sort_values(ascending=False)
    missing_cnt = df.isna().sum().sort_values(ascending=False)

    duplicate_rows = int(df.duplicated().sum())

    nunique_all = df.nunique(dropna=False)
    constant_cols = nunique_all[nunique_all <= 1].index.tolist()

    high_missing_cols = missing_pct[missing_pct >= high_missing_thr].index.tolist()

    id_like_cols = detect_identifier_like_columns(df)

    return {
        "n_rows": int(n_rows),
        "n_cols": int(n_cols),
        "missing_pct": {k: float(v) for k, v in missing_pct.to_dict().items()},
        "missing_cnt": {k: int(v) for k, v in missing_cnt.to_dict().items()},
        "duplicate_rows": int(duplicate_rows),
        "constant_cols": constant_cols,
        "high_missing_cols": high_missing_cols,
        "id_like_cols": id_like_cols,
        "roles": roles
    }


In [127]:
def pick_categorical_column(df, roles):
    cats = [c for c, r in roles.items() if r == "categorical"]
    if not cats:
        return None

    # prefer moderate-cardinality categorical columns
    candidates = []
    for c in cats:
        u = df[c].nunique(dropna=True)
        if 2 <= u <= MAX_CATEGORICAL_UNIQUES:
            candidates.append((u, c))

    if candidates:
        # choose the one closest to ~10 uniques (nice for plots)
        candidates.sort(key=lambda x: abs(x[0] - 10))
        return candidates[0][1]

    # fallback to lowest unique categorical
    return sorted([(df[c].nunique(dropna=True), c) for c in cats])[0][1]

def categorical_frequency_table(df, col, top_k=15):
    vc = df[col].astype("object").value_counts(dropna=False).head(top_k)
    total = int(df.shape[0]) if df.shape[0] else 1
    tab = pd.DataFrame({
        "value": vc.index.astype(str),
        "count": vc.values.astype(int),
        "percent": (vc.values / total * 100).round(2)
    })
    return tab

def pick_numeric_columns(df, roles):
    return [c for c, r in roles.items() if r == "numeric"]

def numeric_profile(df, col):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if len(s) == 0:
        return None

    q1 = float(s.quantile(0.25))
    q3 = float(s.quantile(0.75))
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outliers = int(((s < lo) | (s > hi)).sum())

    mode_val = s.mode()
    mode_val = float(mode_val.iloc[0]) if len(mode_val) else None

    return {
        "col": col,
        "count_nonnull": int(s.shape[0]),
        "min": float(s.min()),
        "max": float(s.max()),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "mode": mode_val,
        "std": float(s.std(ddof=1)) if s.shape[0] > 1 else 0.0,
        "q1": q1,
        "q3": q3,
        "iqr": float(iqr),
        "outliers_1p5iqr": outliers,
        "outlier_bounds": (float(lo), float(hi)),
    }

def numeric_profiles(df, roles, max_cols=2):
    nums = pick_numeric_columns(df, roles)
    chosen = nums[:max_cols]
    profs = []
    for c in chosen:
        p = numeric_profile(df, c)
        if p:
            profs.append(p)
    return profs, chosen


In [128]:
def top_correlations(df, roles, top_n=5):
    nums = pick_numeric_columns(df, roles)
    if len(nums) < 2:
        return []

    corr = df[nums].corr(numeric_only=True)
    pairs = []
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            a, b = nums[i], nums[j]
            val = corr.loc[a, b]
            if pd.notna(val):
                pairs.append((abs(val), float(val), a, b))
    pairs.sort(reverse=True)
    # return strongest absolute correlations
    return [{"a": a, "b": b, "corr": v} for _, v, a, b in pairs[:top_n]]

def pick_scatter_pair(corr_pairs, roles, df):
    # Prefer strongest correlation pair, else first two numeric cols
    if corr_pairs:
        return corr_pairs[0]["a"], corr_pairs[0]["b"]
    nums = pick_numeric_columns(df, roles)
    if len(nums) >= 2:
        return nums[0], nums[1]
    return None, None


In [129]:
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def save_missingness_plot(quality, plot_dir, dataset_name):
    miss = pd.Series(quality["missing_pct"]).sort_values(ascending=False)
    miss = miss[miss > 0]
    if miss.empty:
        # still create a plot to show "no missing"
        plt.figure(figsize=(8,4))
        plt.title(f"Missingness by Column ({dataset_name})")
        plt.xlabel("Columns")
        plt.ylabel("Missing %")
        plt.text(0.5, 0.5, "No missing values detected", ha="center", va="center")
        plt.tight_layout()
        plt.savefig(plot_dir / "missingness.png", dpi=150)
        plt.close()
        return

    plt.figure(figsize=(10,5))
    plt.bar(miss.index.astype(str), miss.values * 100)
    plt.title(f"Missingness by Column ({dataset_name})")
    plt.xlabel("Column")
    plt.ylabel("Missing %")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(plot_dir / "missingness.png", dpi=150)
    plt.close()

def save_histogram(df, col, plot_dir):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        return
    plt.figure(figsize=(8,5))
    plt.hist(s, bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(plot_dir / f"hist_{col}.png", dpi=150)
    plt.close()

def save_boxplot(df, col, plot_dir):
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        return
    plt.figure(figsize=(7,5))
    plt.boxplot(s, vert=True)
    plt.title(f"Boxplot of {col}")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(plot_dir / f"box_{col}.png", dpi=150)
    plt.close()

def save_bar_topcats(freq_table, col, plot_dir):
    if freq_table is None or freq_table.empty:
        return
    plt.figure(figsize=(10,5))
    plt.bar(freq_table["value"], freq_table["count"])
    plt.title(f"Top Categories in {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(plot_dir / f"bar_{col}.png", dpi=150)
    plt.close()

def save_scatter(df, x, y, plot_dir):
    if not x or not y:
        return
    x_s = pd.to_numeric(df[x], errors="coerce")
    y_s = pd.to_numeric(df[y], errors="coerce")
    mask = x_s.notna() & y_s.notna()
    if mask.sum() < 3:
        return
    plt.figure(figsize=(7,5))
    plt.scatter(x_s[mask], y_s[mask], s=12)
    plt.title(f"Scatterplot: {x} vs {y}")
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.savefig(plot_dir / f"scatter_{x}_vs_{y}.png", dpi=150)
    plt.close()

def save_missingness_profile(df, plot_dir, top_n=15):
    """
    Plot missing percentage for top-N columns with highest missingness.
    Always produces a plot (even if missingness is zero).
    """
    missing_pct = df.isna().mean() * 100
    missing_pct = missing_pct.sort_values(ascending=False).head(top_n)

    plt.figure(figsize=(10,5))
    plt.bar(missing_pct.index.astype(str), missing_pct.values)
    plt.title(f"Missingness Profile (Top {top_n} Columns)")
    plt.xlabel("Column")
    plt.ylabel("Missing Percentage (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(plot_dir / "missingness_profile.png", dpi=150)
    plt.close()


def save_numeric_by_category(df, cat_col, num_col, plot_dir):
    if not cat_col or not num_col:
        return
    tmp = df[[cat_col, num_col]].dropna()
    if tmp.empty:
        return
    # keep only top categories to avoid huge plots
    top = tmp[cat_col].value_counts().head(TOP_K_CATS).index
    tmp = tmp[tmp[cat_col].isin(top)]
    if tmp.empty:
        return
    grouped = tmp.groupby(cat_col)[num_col].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,5))
    plt.bar(grouped.index.astype(str), grouped.values)
    plt.title(f"Mean {num_col} by {cat_col} (Top {TOP_K_CATS})")
    plt.xlabel(cat_col)
    plt.ylabel(f"Mean {num_col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(plot_dir / f"mean_{num_col}_by_{cat_col}.png", dpi=150)
    plt.close()

def make_plots(df, roles, quality, dataset_name, out_base: Path):
    plot_dir = ensure_dir(out_base / dataset_name / "plots")

    # 1️⃣ Missingness summary plot (counts as one)
    save_missingness_plot(quality, plot_dir, dataset_name)

    # 2️⃣ Missingness profile (replacement for correlation heatmap)
    save_missingness_profile(df, plot_dir, top_n=TOP_K_CATS)

    # 3️⃣ Numeric plots (hist + box for first numeric)
    nums = pick_numeric_columns(df, roles)
    if nums:
        save_histogram(df, nums[0], plot_dir)
        save_boxplot(df, nums[0], plot_dir)

    # 4️⃣ Categorical bar plot
    cat_col = pick_categorical_column(df, roles)
    if cat_col:
        freq_table = categorical_frequency_table(df, cat_col, top_k=TOP_K_CATS)
        save_bar_topcats(freq_table, cat_col, plot_dir)

    # 5️⃣ Scatterplot (if possible)
    corr_pairs = top_correlations(df, roles, top_n=5)
    x, y = pick_scatter_pair(corr_pairs, roles, df)
    save_scatter(df, x, y, plot_dir)

    # 6️⃣ Optional: numeric by category (extra)
    if cat_col and nums:
        save_numeric_by_category(df, cat_col, nums[0], plot_dir)

    return plot_dir



In [130]:
def build_facts_bundle(dataset_name, df, roles, quality, schema_obj):
    # Categorical
    cat_col = pick_categorical_column(df, roles)
    cat_table = categorical_frequency_table(df, cat_col, top_k=TOP_K_CATS) if cat_col else None

    # Numeric profiles (2 columns if available)
    num_profs, chosen_nums = numeric_profiles(df, roles, max_cols=2)

    # Correlations
    corr_pairs = top_correlations(df, roles, top_n=5)

    # Schema snippets (if any)
    schema_info = None
    if schema_obj:
        schema_info = schema_obj

    facts = {
        "dataset_name": dataset_name,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "overview": {
            "n_rows": quality["n_rows"],
            "n_cols": quality["n_cols"],
            "columns": list(df.columns),
            "roles": roles,
        },
        "quality": {
            "duplicate_rows": quality["duplicate_rows"],
            "constant_cols": quality["constant_cols"],
            "high_missing_threshold": HIGH_MISSING_THRESHOLD,
            "high_missing_cols": quality["high_missing_cols"],
            "id_like_cols": quality["id_like_cols"],
            "missing_pct_top": dict(list(sorted(quality["missing_pct"].items(), key=lambda x: x[1], reverse=True))[:10]),
        },
        "categorical": {
            "selected_col": cat_col,
            "frequency_top": cat_table.to_dict(orient="records") if cat_table is not None else [],
        },
        "numeric": {
            "selected_cols": chosen_nums,
            "profiles": num_profs,
        },
        "correlations": corr_pairs,
        "schema": schema_info,
    }
    return facts, cat_table


In [131]:
USE_LLM = True  # set False if you don't want API calls

def llm_generate_insights_from_facts(facts: dict):
    """
    IMPORTANT: LLM must never invent numbers.
    We give it computed facts and demand JSON output.
    If API key isn't present, raise so caller can fallback.
    """
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY not set. Falling back to deterministic insights.")

    # OpenAI SDK (if not installed: pip install openai)
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    # Keep prompt compact and enforce "use only provided facts".
    system = (
        "You are a data analyst writing EDA insights. "
        "Use ONLY the provided computed facts. "
        "Do NOT infer or fabricate numeric values. "
        "Return JSON with keys: insights (5-10 bullets), limitations (list of bullets)."
    )

    user = {
        "task": "Generate specific insights and multiple limitations from EDA facts.",
        "facts": facts
    }

    # Use a modern model you have access to; if your account allows, keep it stable.
    resp = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": json.dumps(user)}
        ],
        # We ask for JSON in plain text; strict JSON parsing below.
    )

    text = resp.output_text.strip()
    # Parse JSON safely
    data = json.loads(text)
    return data


In [132]:
def deterministic_insights_and_limitations(facts: dict):
    insights = []
    limitations = []

    ov = facts["overview"]
    q = facts["quality"]
    cat = facts["categorical"]
    num = facts["numeric"]
    corr = facts["correlations"]

    # Insight 1: dataset size
    insights.append(f"Dataset has {ov['n_rows']} rows and {ov['n_cols']} columns.")

    # Missingness insight
    miss_top = q.get("missing_pct_top", {})
    if miss_top:
        top_col, top_pct = next(iter(sorted(miss_top.items(), key=lambda x: x[1], reverse=True)))
        if top_pct > 0:
            insights.append(f"Highest missingness is in '{top_col}' at {top_pct*100:.2f}% of rows.")
        else:
            insights.append("No missing values detected across columns (0% missingness).")

    # High missing cols
    if q["high_missing_cols"]:
        insights.append(f"High-missing columns (≥{q['high_missing_threshold']*100:.0f}% missing): {', '.join(q['high_missing_cols'])}.")
        limitations.append(f"High missingness in {', '.join(q['high_missing_cols'])} may bias summaries/models if not handled (imputation, exclusion, or missingness modeling).")
    else:
        insights.append(f"No columns exceed the high-missing threshold ({q['high_missing_threshold']*100:.0f}%).")

    # Duplicates
    if q["duplicate_rows"] > 0:
        insights.append(f"Found {q['duplicate_rows']} duplicate rows (consider deduplication).")
        limitations.append("Duplicate rows can inflate counts/statistics if not removed or justified.")
    else:
        insights.append("No duplicate rows detected.")

    # Constant columns
    if q["constant_cols"]:
        insights.append(f"Constant / single-value columns detected: {', '.join(q['constant_cols'])}.")
        limitations.append("Constant columns provide no predictive value and may be artifacts of data collection or filtering.")
    else:
        insights.append("No constant (single-value) columns detected.")

    # Identifier-like columns
    if q["id_like_cols"]:
        insights.append(f"Possible identifier-like columns detected (high uniqueness): {', '.join(q['id_like_cols'])}.")
        limitations.append("Identifier-like columns may contain sensitive info and can cause leakage in modeling; avoid exposing raw values in reports.")
    else:
        insights.append("No strong identifier-like columns detected by uniqueness/name heuristics (still review for privacy).")

    # Categorical insight
    if cat["selected_col"] and cat["frequency_top"]:
        top = cat["frequency_top"][0]
        insights.append(
            f"In '{cat['selected_col']}', the most common value is '{top['value']}' "
            f"with {top['count']} rows ({top['percent']}%)."
        )
        # Limitations: if many categories
        uniq_est = len(set([r["value"] for r in cat["frequency_top"]]))
        if uniq_est >= TOP_K_CATS:
            limitations.append(f"Categorical column '{cat['selected_col']}' may have many levels; plotting/truncation to top {TOP_K_CATS} can hide long-tail categories.")
    else:
        limitations.append("No suitable categorical column found (or all categorical fields are high-cardinality text).")

    # Numeric insights
    if num["profiles"]:
        for p in num["profiles"]:
            insights.append(
                f"Numeric '{p['col']}': mean={p['mean']:.3f}, median={p['median']:.3f}, "
                f"std={p['std']:.3f}, IQR={p['iqr']:.3f}, outliers(1.5×IQR)={p['outliers_1p5iqr']}."
            )
            if p["outliers_1p5iqr"] > 0:
                lo, hi = p["outlier_bounds"]
                limitations.append(
                    f"Outliers in '{p['col']}' beyond [{lo:.3f}, {hi:.3f}] may skew mean; consider robust stats or winsorization."
                )
    else:
        limitations.append("No numeric columns available for numeric descriptive statistics.")

    # Correlation insights
    if corr:
        strongest = corr[0]
        insights.append(
            f"Strongest numeric correlation observed: corr({strongest['a']}, {strongest['b']}) = {strongest['corr']:.3f}."
        )
        limitations.append("Correlation does not imply causation; observed relationships may be confounded.")
    else:
        limitations.append("Not enough numeric columns to compute correlations/heatmap.")

    # Datetime limitations
    roles = ov["roles"]
    if any(r == "datetime" for r in roles.values()):
        limitations.append("Datetime parsing is heuristic-based; some date-like strings may be misparsed or dropped if inconsistent.")

    # Sampling / coverage generic but still real
    limitations.append("This EDA reflects only the provided dataset; sampling method, population coverage, and time window may limit generalization.")

    # Ensure insight count 5–10
    insights = insights[:10] if len(insights) > 10 else insights
    # Ensure at least 5 insights
    while len(insights) < 5:
        insights.append("Additional insights require more numeric/categorical fields or richer schema context.")

    # Deduplicate limitations
    limitations = list(dict.fromkeys(limitations))

    return {"insights": insights, "limitations": limitations}


In [133]:
def write_report(dataset_name, out_base: Path, facts: dict, cat_table: pd.DataFrame | None,
                 insights_block: dict, plot_dir: Path):
    ds_dir = out_base / dataset_name
    ds_dir.mkdir(parents=True, exist_ok=True)

    # Save summary JSON
    (ds_dir / "summary.json").write_text(json.dumps(facts, indent=2), encoding="utf-8")

    # Roles table
    roles_dict = facts.get("overview", {}).get("roles", {})
    roles_df = pd.DataFrame({"column": list(roles_dict.keys()),
                             "role": list(roles_dict.values())})

    # ✅ Safe missingness fetch (prevents KeyError)
    quality = facts.get("quality", {})
    missing_cnt = quality.get("missing_cnt", {})
    missing_pct = quality.get("missing_pct", {})

    missing_df = pd.DataFrame({
        "missing_count": pd.Series(missing_cnt, dtype="int64"),
        "missing_pct": (pd.Series(missing_pct, dtype="float64") * 100).round(2)
    })
    if not missing_df.empty:
        missing_df = missing_df.sort_values("missing_pct", ascending=False)

    numeric_profiles = facts.get("numeric", {}).get("profiles", [])
    numeric_df = pd.DataFrame(numeric_profiles) if numeric_profiles else pd.DataFrame()

    # Markdown report
    lines = []
    lines.append(f"# EDA Report — {dataset_name}")
    lines.append(f"_Generated: {facts.get('generated_at','')}_\n")

    # Schema note
    if facts.get("schema"):
        lines.append("## Schema / Data Dictionary")
        lines.append("Schema provided and loaded. (See summary.json for full parsed schema.)\n")

    # Overview
    ov = facts.get("overview", {})
    lines.append("## Dataset Overview")
    lines.append(f"- Rows: **{ov.get('n_rows','?')}**")
    lines.append(f"- Columns: **{ov.get('n_cols','?')}**\n")

    lines.append("### Column Roles (inferred)")
    lines.append(roles_df.to_markdown(index=False) if not roles_df.empty else "_No columns found._")
    lines.append("")

    # Quality checks
    lines.append("## Data Quality Checks")
    lines.append(f"- Duplicate rows: **{quality.get('duplicate_rows','?')}**")
    const_cols = quality.get("constant_cols", [])
    lines.append(f"- Constant (single-value) columns: **{', '.join(const_cols) if const_cols else 'None'}**")
    hm_thr = quality.get("high_missing_threshold", HIGH_MISSING_THRESHOLD)
    hm_cols = quality.get("high_missing_cols", [])
    lines.append(f"- High-missing columns (≥{float(hm_thr)*100:.0f}%): **{', '.join(hm_cols) if hm_cols else 'None'}**")
    id_cols = quality.get("id_like_cols", [])
    lines.append(f"- Possible identifier-like columns: **{', '.join(id_cols) if id_cols else 'None'}**\n")

    lines.append("### Missingness Summary (count and %)")
    lines.append(missing_df.to_markdown() if not missing_df.empty else "_Missingness summary unavailable._")
    lines.append("")

    # Descriptive statistics
    lines.append("## Descriptive Statistics")

    # Categorical required
    cat = facts.get("categorical", {})
    lines.append("### Categorical Column Frequency (counts + %)")
    if cat.get("selected_col") and cat_table is not None and not cat_table.empty:
        lines.append(f"Selected column: **{cat['selected_col']}**")
        lines.append(cat_table.to_markdown(index=False))
    else:
        lines.append("_No suitable categorical column found._")
    lines.append("")

    # Numeric required
    lines.append("### Numeric Column Statistics (min/max/mean/median/mode/std/IQR/outliers)")
    lines.append(numeric_df.to_markdown(index=False) if not numeric_df.empty else "_No numeric columns found._")
    lines.append("")

    # Visualizations
    lines.append("## Visualizations")
    lines.append("Plots saved to `plots/`:\n")
    pngs = sorted(Path(plot_dir).glob("*.png"))
    for p in pngs:
        lines.append(f"- {p.name}")
    lines.append("")

    # Insights
    lines.append("## Insights (computed + narrative)")
    for b in insights_block.get("insights", []):
        lines.append(f"- {b}")
    lines.append("")

    # Limitations
    lines.append("## Limitations / Potential Biases")
    for b in insights_block.get("limitations", []):
        lines.append(f"- {b}")
    lines.append("")

    report_path = ds_dir / "report.md"
    report_path.write_text("\n".join(lines), encoding="utf-8")
    return report_path


In [134]:
def run_full_eda(csv_path: str, dataset_name: str, schema_path: str | None = None, out_base: Path = OUTDIR):
    # Load schema (optional)
    schema_obj = load_schema(schema_path)
    extra_missing = schema_missing_codes(schema_obj)

    # Read + missing normalization
    df = read_csv_robust(csv_path)
    df = apply_missing_codes(df, extra_missing_codes=extra_missing)

    # Infer roles
    roles, df = infer_column_roles(df)

    # Quality checks
    quality = dataset_quality_checks(df, roles, high_missing_thr=HIGH_MISSING_THRESHOLD)

    # Facts bundle
    facts, cat_table = build_facts_bundle(dataset_name, df, roles, quality, schema_obj)

    # Plots (>=5)
    plot_dir = make_plots(df, roles, quality, dataset_name, out_base)

    # Insights (LLM if configured, else deterministic)
    if USE_LLM:
        try:
            insights_block = llm_generate_insights_from_facts(facts)
            # minimal validation of structure
            if not isinstance(insights_block, dict) or "insights" not in insights_block or "limitations" not in insights_block:
                insights_block = deterministic_insights_and_limitations(facts)
        except Exception as e:
            print(f"[{dataset_name}] LLM unavailable/failure -> deterministic insights. Reason:", str(e))
            insights_block = deterministic_insights_and_limitations(facts)
    else:
        insights_block = deterministic_insights_and_limitations(facts)

    # Force insight count 5–10
    insights_block["insights"] = insights_block["insights"][:10]
    if len(insights_block["insights"]) < 5:
        while len(insights_block["insights"]) < 5:
            insights_block["insights"].append("Not enough signal for additional specific insights (limited numeric/categorical richness).")

    # Ensure multiple limitations (not just 1)
    insights_block["limitations"] = list(dict.fromkeys(insights_block["limitations"]))  # unique
    if len(insights_block["limitations"]) < 2:
        insights_block["limitations"].append("Limited metadata about sampling/collection restricts bias assessment.")

    # Write report + summary.json
    report_path = write_report(dataset_name, out_base, facts, cat_table, insights_block, plot_dir)

    # Count plots for rubric
    n_plots = len(list(plot_dir.glob("*.png")))
    return {
        "dataset_name": dataset_name,
        "report_path": str(report_path),
        "plot_dir": str(plot_dir),
        "n_plots": n_plots,
        "facts": facts,
        "insights": insights_block
    }


In [135]:
results = []
for d in DATASETS:
    res = run_full_eda(d["csv"], d["name"], d.get("schema"))
    results.append(res)
    print(f"✅ {d['name']} done | plots={res['n_plots']} | report={res['report_path']}")

# Quick rubric check:
for r in results:
    assert r["n_plots"] >= 5, f"{r['dataset_name']} has <5 plots!"
print("✅ Rubric check: all datasets have >=5 plots.")


/tmp/ipykernel_429/193362287.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_429/193362287.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)


[data2] LLM unavailable/failure -> deterministic insights. Reason: OPENAI_API_KEY not set. Falling back to deterministic insights.
✅ data2 done | plots=7 | report=output/data2/report.md


/tmp/ipykernel_429/193362287.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_429/193362287.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors="coerce", infer_datetime_format=True)
/tmp/ipykernel_429/193362287.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, error

[chocolatesales] LLM unavailable/failure -> deterministic insights. Reason: OPENAI_API_KEY not set. Falling back to deterministic insights.
✅ chocolatesales done | plots=6 | report=output/chocolatesales/report.md
✅ Rubric check: all datasets have >=5 plots.
